# 03y_dim_player_alias

**Purpose:** Persistent decision/alias table so the fuzzy-review process never
re-asks about a player already resolved. A transformer in spirit, like
`dim_position` / `dim_school` — but for player-name resolution.

**Problem it solves:** a name variant X (e.g. "Omar Cooper Jr.") fuzzy-matches an
existing prospect Y ("Omar Cooper") at 70-89. Saying `match` previously recorded
nothing, so every later run re-surfaced X for review AND `ingest_ranking_source`
dropped X's ranking (its clean name was never in `dim_rookie_prospect`). The alias
maps X -> Y's `player_key` once, permanently.

**Consumed by:**
- matchers (03a-03x `add_players_from_source`): if `(name_clean, position_raw)` is
  in the alias, skip review.
- ingest (03a-03x `ingest_ranking_source`): augment `name_to_key` with the alias so
  matched variants attribute their ranking to the resolved `player_key`.
- apply (03z): append every `match`/`new` decision here.

**Key:** `(name_clean, position_raw)`.

**Outputs:** `data/dim_player_alias.parquet`


In [1]:
# ---- Setup & Config ---------------------------------------------------------
import glob
import hashlib
from dataclasses import dataclass
from datetime import date
from pathlib import Path

import pandas as pd


@dataclass
class LeagueConfig:
    """Central config."""
    draft_year: int = 2026
    data_dir: str = "data"
    review_dir: str = "data/review"
    alias_name: str = "dim_player_alias"


CFG = LeagueConfig()
DATA = Path(CFG.data_dir)
REVIEW = Path(CFG.review_dir)
TODAY = date.today().isoformat()


def clean_player_name(name: str) -> str:
    # Matches the helper in 03x / 03z exactly.
    if pd.isna(name):
        return ""
    s = str(name).strip().replace(".", "").replace("\u00a0", " ")
    s = s.replace("\u2018", "'").replace("\u2019", "'").replace("`", "'")
    return " ".join(s.split()).lower()


def generate_player_key(name: str, position: str, school: str) -> str:
    raw = f"{clean_player_name(name)}|{str(position).upper().strip()}|{str(school).strip().lower()}"
    return hashlib.md5(raw.encode()).hexdigest()[:12]


## Backfill from archived review decisions

Reads every `review_fuzzy_matches.applied_*.csv` and folds the resolved decisions
into the alias. `match` -> target is `best_match_key`; `new` -> the generated
`player_key` for the added prospect. Latest decision wins on conflict.

In [2]:
# ---- Backfill ---------------------------------------------------------------
ALIAS_COLS = ["name_clean", "position_raw", "player_key",
              "decision", "source_example", "decided_date"]


def _decision_rows(df: pd.DataFrame, decided_date: str) -> list[dict]:
    rows = []
    for _, r in df.iterrows():
        action = str(r.get("action", "")).strip().lower()
        name_clean = r.get("new_name_clean")
        if pd.isna(name_clean) or not str(name_clean).strip():
            name_clean = clean_player_name(r.get("new_name", ""))
        pos = str(r.get("new_position", "")).upper().strip()
        if action == "match":
            pkey = r.get("best_match_key")
        elif action == "new":
            pkey = generate_player_key(r.get("new_name", ""), pos, r.get("new_school", ""))
        else:
            continue
        if pd.isna(pkey) or not str(pkey).strip():
            continue
        rows.append({
            "name_clean": str(name_clean), "position_raw": pos,
            "player_key": str(pkey), "decision": action,
            "source_example": r.get("source", ""), "decided_date": decided_date,
        })
    return rows


backfill = []
for f in sorted(glob.glob(str(REVIEW / "review_fuzzy_matches.applied_*.csv"))):
    stamp = Path(f).stem.split("applied_")[-1]              # e.g. 20260520
    dd = f"{stamp[:4]}-{stamp[4:6]}-{stamp[6:8]}" if len(stamp) == 8 else TODAY
    rows = _decision_rows(pd.read_csv(f), dd)
    backfill.extend(rows)
    print(f"{Path(f).name}: +{len(rows)} decisions")

alias = pd.DataFrame(backfill, columns=ALIAS_COLS)
# latest decided_date wins for a given (name_clean, position_raw)
alias = (alias.sort_values("decided_date")
              .drop_duplicates(subset=["name_clean", "position_raw"], keep="last")
              .reset_index(drop=True))
alias.to_parquet(DATA / f"{CFG.alias_name}.parquet", index=False)
print(f"\n[ok] dim_player_alias: {len(alias)} rows -> data/{CFG.alias_name}.parquet")
print(alias["decision"].value_counts().to_string())


review_fuzzy_matches.applied_20260513.csv: +15 decisions
review_fuzzy_matches.applied_20260515.csv: +20 decisions
review_fuzzy_matches.applied_20260520.csv: +67 decisions

[ok] dim_player_alias: 35 rows -> data/dim_player_alias.parquet
decision
match    19
new      16


In [3]:
# ---- Validation -------------------------------------------------------------
alias = pd.read_parquet(DATA / f"{CFG.alias_name}.parquet")
print(f"alias rows: {len(alias)} | unique name_clean: {alias['name_clean'].nunique()}")
print()
# every match target must exist in dim_rookie_prospect
dim_rp = pd.read_parquet(DATA / "dim_rookie_prospect.parquet")
valid_keys = set(dim_rp["player_key"])
bad = alias[(alias["decision"] == "match") & (~alias["player_key"].isin(valid_keys))]
if len(bad):
    print(f"[warn] {len(bad)} match aliases point at a missing player_key:")
    print(bad[["name_clean", "player_key", "source_example"]].to_string())
else:
    print("[ok] all match targets exist in dim_rookie_prospect")
print()
print(alias.head(12).to_string())


alias rows: 35 | unique name_clean: 32

[ok] all match targets exist in dim_rookie_prospect

        name_clean position_raw    player_key decision         source_example decided_date
0   rueben bain jr           DE  f35fadfc88cc    match      WalterFootball_DE   2026-05-20
1      rara thomas           WR  c469ee117477      new      WalterFootball_WR   2026-05-20
2      jayce brown           WR  44d6258be965      new      WalterFootball_WR   2026-05-20
3       cian slone           DE  d953feaf4af9      new      WalterFootball_DE   2026-05-20
4   riley williams           TE  236ecb45b5f6      new  FantasyPros_Superflex   2026-05-20
5    darius taylor           RB  0d9e70f3151f      new      WalterFootball_RB   2026-05-20
6       lt overton           DE  190456f5a438    match    WalterFootball_DE34   2026-05-20
7      byron eason           DT  e0199fb01ed6    match      WalterFootball_DT   2026-05-20
8   rueben bain jr          OLB  f35fadfc88cc    match   WalterFootball_OLB34   2026-05-